# Titanic Survival Prediction

Goal: predict passenger survival using Titanic passenger attributes. This notebook covers EDA, feature engineering, missing value handling, model comparison, explainability, and inference.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

import sys
sys.path.append('../src')
from preprocessing import TitanicFeatureEngineer

ROOT = Path('..')
DATA_PATH = ROOT / 'data' / 'titanic.csv'
PLOTS_DIR = ROOT / 'plots'
MODELS_DIR = ROOT / 'models'
OUTPUTS_DIR = ROOT / 'outputs'
for d in [PLOTS_DIR, MODELS_DIR, OUTPUTS_DIR]:
    d.mkdir(exist_ok=True)

## 2. Load Dataset and Basic Checks

In [ ]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
display(df.head())
display(df.info())
display(df.isna().sum())
display(df.describe(include='all'))

## 3. EDA and Class Separability

In [ ]:
plt.figure(figsize=(6,4))
df['Survived'].value_counts().sort_index().plot(kind='bar')
plt.title('Survival Class Distribution')
plt.xlabel('Survived (0 = No, 1 = Yes)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,4))
df.groupby('Sex')['Survived'].mean().plot(kind='bar')
plt.title('Survival Rate by Sex')
plt.ylabel('Survival Rate')
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,4))
df.groupby('Pclass')['Survived'].mean().plot(kind='bar')
plt.title('Survival Rate by Passenger Class')
plt.ylabel('Survival Rate')
plt.tight_layout()
plt.show()

## 4. Feature Engineering

Features created: `Title`, `FamilySize`, `IsAlone`, `CabinPresent`, `Deck`, and `FareLog`.

In [ ]:
fe_df = TitanicFeatureEngineer().fit_transform(df)
display(fe_df[['Name','Title','SibSp','Parch','FamilySize','IsAlone','Cabin','CabinPresent','Deck','Fare','FareLog']].head())

plt.figure(figsize=(8,4))
fe_df.groupby('Title')['Survived'].mean().sort_values(ascending=False).plot(kind='bar')
plt.title('Survival Rate by Extracted Title')
plt.ylabel('Survival Rate')
plt.tight_layout()
plt.show()

## 5. Preprocessing Pipeline

Missing numeric values are handled using median imputation. Missing categorical values are handled using most frequent imputation. Categorical variables are one-hot encoded and numeric variables are scaled.

In [ ]:
numeric_features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FareLog', 'FamilySize', 'IsAlone', 'CabinPresent']
categorical_features = ['Sex', 'Embarked', 'Title', 'Deck']

numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipe, numeric_features),
    ('cat', categorical_pipe, categorical_features)
])

def make_pipeline(model):
    return Pipeline([
        ('feature_engineering', TitanicFeatureEngineer()),
        ('preprocessor', preprocessor),
        ('model', model)
    ])

## 6. Train and Compare Classification Models

In [ ]:
X = df.drop(columns=['Survived'])
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=5, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

rows = []
fitted_models = {}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    pipe = make_pipeline(model)
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='accuracy')
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    rows.append({
        'model': name,
        'cv_accuracy_mean': cv_scores.mean(),
        'cv_accuracy_std': cv_scores.std(),
        'test_accuracy': accuracy_score(y_test, preds),
        'precision': precision_score(y_test, preds, zero_division=0),
        'recall': recall_score(y_test, preds, zero_division=0),
        'f1': f1_score(y_test, preds, zero_division=0)
    })
    fitted_models[name] = pipe

metrics_df = pd.DataFrame(rows).sort_values(['f1','test_accuracy'], ascending=False)
display(metrics_df)
metrics_df.to_csv(OUTPUTS_DIR / 'model_comparison_metrics.csv', index=False)

## 7. Final Metrics and Confusion Matrix

In [ ]:
best_name = metrics_df.iloc[0]['model']
best_model = fitted_models[best_name]
y_pred = best_model.predict(X_test)
print('Best model:', best_name)
print(classification_report(y_test, y_pred, zero_division=0))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Not Survived', 'Survived']).plot(values_format='d')
plt.title(f'Confusion Matrix - {best_name}')
plt.tight_layout()
plt.show()

## 8. Model Explainability using Feature Importance

In [ ]:
feature_names = best_model.named_steps['preprocessor'].get_feature_names_out()
final_model = best_model.named_steps['model']

if hasattr(final_model, 'feature_importances_'):
    importance = final_model.feature_importances_
elif hasattr(final_model, 'coef_'):
    importance = np.abs(final_model.coef_[0])
else:
    importance = np.zeros(len(feature_names))

imp = pd.DataFrame({'feature': feature_names, 'importance': importance}).sort_values('importance', ascending=False).head(15)
display(imp)

plt.figure(figsize=(8,6))
plt.barh(imp['feature'][::-1], imp['importance'][::-1])
plt.title(f'Top Feature Importance - {best_name}')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 9. Save Best Model

In [ ]:
joblib.dump(best_model, MODELS_DIR / 'best_titanic_survival_model.joblib')
print('Saved:', MODELS_DIR / 'best_titanic_survival_model.joblib')

## 10. Example Inference Cell

In [ ]:
loaded_model = joblib.load(MODELS_DIR / 'best_titanic_survival_model.joblib')

sample_passenger = pd.DataFrame([{
    'PassengerId': 10001,
    'Pclass': 3,
    'Name': 'Doe, Mr. John',
    'Sex': 'male',
    'Age': 28,
    'SibSp': 1,
    'Parch': 0,
    'Ticket': 'A/5 21171',
    'Fare': 7.25,
    'Cabin': None,
    'Embarked': 'S'
}])

pred = loaded_model.predict(sample_passenger)[0]
prob = loaded_model.predict_proba(sample_passenger)[0][1]
print('Prediction:', 'Survived' if pred == 1 else 'Not Survived')
print('Survival probability:', round(float(prob), 4))